# Pré-processamento CelebA-HQ — detecção + alinhamento

Usa `models/face_detector.py` (SCRFD 2.5G KPS) para detectar cada face,
alinhar pelos 5 landmarks e salvar imagens 224×224 prontas para `data/dataset.py`.

In [ ]:
import os
import sys
from pathlib import Path

import cv2
from insightface.utils import face_align
from PIL import Image
from tqdm import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.face_detector import get_default_detector, _to_bgr_uint8

INPUT_DIR = "/mnt/study-data/dcarvalho/datasets/celebahq"
OUTPUT_DIR = "/mnt/study-data/dcarvalho/datasets/celebahq_pp"
IMAGE_SIZE = 224
DET_THRESH = 0.5
CTX_ID = -1  # -1 = CPU; 0 = GPU

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Entrada:  {INPUT_DIR}")
print(f"Saída:    {OUTPUT_DIR}")
print(f"Tamanho:  {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Threshold: {DET_THRESH}")

In [ ]:
detector = get_default_detector(ctx_id=CTX_ID, det_thresh=DET_THRESH)

input_files = sorted(
    p for p in Path(INPUT_DIR).glob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)
print(f"Imagens encontradas: {len(input_files)}")

In [ ]:
saved = 0
discarded_no_face = 0
discarded_low_score = 0
discarded_read_error = 0

for src_path in tqdm(input_files, desc="Pré-processando"):
    try:
        bgr = _to_bgr_uint8(src_path)
    except (FileNotFoundError, OSError):
        discarded_read_error += 1
        continue

    detection = detector.detect_best(src_path)
    if detection is None:
        discarded_no_face += 1
        continue
    if detection.score < DET_THRESH:
        discarded_low_score += 1
        continue

    aligned_bgr = face_align.norm_crop(bgr, detection.landmarks, image_size=IMAGE_SIZE)
    aligned_rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
    out_path = Path(OUTPUT_DIR) / f"{src_path.stem}.jpg"
    Image.fromarray(aligned_rgb).save(out_path, format="JPEG", quality=95)
    saved += 1

total = len(input_files)
discarded = discarded_no_face + discarded_low_score + discarded_read_error

print(f"\nConcluído!")
print(f"  Total de entrada:     {total}")
print(f"  Salvas:               {saved}")
print(f"  Descartadas (total):  {discarded}")
print(f"    sem face:           {discarded_no_face}")
print(f"    score < {DET_THRESH}:      {discarded_low_score}")
print(f"    erro de leitura:    {discarded_read_error}")
print(f"  Taxa de descarte:     {100.0 * discarded / max(total, 1):.2f}%")
print(f"\nDataset salvo em: {OUTPUT_DIR}")

In [ ]:
from data.dataset import FaceImageFolder
import random

dataset = FaceImageFolder(OUTPUT_DIR, image_size=IMAGE_SIZE)
print(f"FaceImageFolder carregou {len(dataset)} imagens.")

idx = random.randint(0, len(dataset) - 1)
tensor, path = dataset[idx]
print(f"Exemplo: {path} | shape={tuple(tensor.shape)}")

from IPython.display import display
display(Image.open(path))